# Analisis & Persiapan Data Gizi Makanan Indonesia - Progress 2
Dataset: `nutrition.csv`

In [4]:
# ================================================================
# ANALISIS & PERSIAPAN DATA GIZI MAKANAN INDONESIA - PROGRESS 2
# Capaian 6-14: EDA, audit kualitas, cleaning, preprocessing, feature selection
# Dataset: nutrition.csv (letakkan satu folder dengan file program ini)
# ================================================================

from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import mutual_info_classif, f_classif
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / "nutrition.csv"
OUT_DIR = BASE_DIR / "hasil_progress2"
OUT_DIR.mkdir(exist_ok=True)

NUM_COLS = ["calories", "proteins", "fat", "carbohydrate"]
MACRO_COLS = ["proteins", "fat", "carbohydrate"]
TARGET_ORDER = ["Rendah", "Sedang", "Tinggi"]
TARGET_MAP = {"Rendah": 0, "Sedang": 1, "Tinggi": 2}


def save_plot(filename):
    plt.tight_layout()
    plt.savefig(OUT_DIR / filename, dpi=250, bbox_inches="tight")
    plt.close()


def kategori_kalori(kalori):
    """Batas kelas yang digunakan dalam proyek: <150, 150-300, >300 kkal."""
    if pd.isna(kalori):
        return np.nan
    if kalori < 150:
        return "Rendah"
    if kalori <= 300:
        return "Sedang"
    return "Tinggi"


## 0. LOAD DATA DAN VALIDASI STRUKTUR

In [5]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset tidak ditemukan: {DATA_PATH}\n"
        "Simpan nutrition.csv dalam folder yang sama dengan file .py ini."
    )

df = pd.read_csv(DATA_PATH)
required = ["id", "calories", "proteins", "fat", "carbohydrate", "name", "image"]
missing_columns = [c for c in required if c not in df.columns]
if missing_columns:
    raise ValueError(f"Kolom wajib tidak ditemukan: {missing_columns}")

# Pastikan kolom nutrisi numerik; teks yang tidak dapat dikonversi menjadi NaN.
for col in NUM_COLS:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Bersihkan spasi pada kolom teks sejak awal.
for col in ["name", "image"]:
    df[col] = df[col].astype("string").str.strip()
    df[col] = df[col].replace("", pd.NA)

print("=" * 65)
print("UKURAN DATA AWAL:", df.shape)
print("=" * 65)
print(df.head())


UKURAN DATA AWAL: (1346, 7)
   id  calories  proteins   fat  carbohydrate                name  \
0   1     280.0       9.2  28.4           0.0                Abon   
1   2     513.0      23.7  37.0          21.3        Abon haruwan   
2   3       0.0       0.0   0.2           0.0           Agar-agar   
3   4      45.0       1.1   0.4          10.8  Akar tonjong segar   
4   5      37.0       4.4   0.5           3.8       Aletoge segar   

                                               image  
0  https://img-cdn.medkomtek.com/PbrY9X3ignQ8sVuj...  
1  https://img-global.cpcdn.com/recipes/cbf330fbd...  
2  https://res.cloudinary.com/dk0z4ums3/image/upl...  
3  https://images.tokopedia.net/img/cache/200-squ...  
4  https://nilaigizi.com/assets/images/produk/pro...  


## 6. EDA: STATISTIK DESKRIPTIF

In [6]:
print("\n[CAPAIAN 6] STATISTIK DESKRIPTIF")
desc = df[NUM_COLS].describe().T
desc["median"] = df[NUM_COLS].median()
desc["skewness"] = df[NUM_COLS].skew()
desc.to_csv(OUT_DIR / "01_statistik_deskriptif.csv")
print(desc.round(2))



[CAPAIAN 6] STATISTIK DESKRIPTIF
               count    mean     std  min    25%    50%     75%    max  \
calories      1346.0  203.22  163.08  0.0  75.00  146.0  333.75  940.0   
proteins      1346.0   10.00   11.85  0.0   1.80    5.0   15.00   83.0   
fat           1346.0    7.58   13.73  0.0   0.50    2.0    8.28  100.0   
carbohydrate  1346.0   25.39   32.19  0.0   4.53   13.3   37.58  647.0   

              median  skewness  
calories       146.0      1.19  
proteins         5.0      2.15  
fat              2.0      3.54  
carbohydrate    13.3      6.54  


## 7. EDA: TARGET DAN VISUALISASI FITUR TERHADAP TARGET

In [7]:
# Target dibuat dari calories. Kolom calories tidak boleh menjadi fitur X
# karena akan menyebabkan target leakage.
df["kategori_kalori"] = df["calories"].apply(kategori_kalori)
print("\n[CAPAIAN 7] DISTRIBUSI TARGET")
print(df["kategori_kalori"].value_counts(dropna=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.countplot(data=df, x="kategori_kalori", order=TARGET_ORDER, color="steelblue", ax=axes[0])
axes[0].set_title("Jumlah Data per Kategori Kalori")
axes[0].set_xlabel("Kategori kalori")
axes[0].set_ylabel("Jumlah makanan")
(df["kategori_kalori"].value_counts(normalize=True)
   .reindex(TARGET_ORDER).mul(100).plot(kind="bar", ax=axes[1]))
axes[1].set_title("Proporsi Kategori Kalori")
axes[1].set_xlabel("Kategori kalori")
axes[1].set_ylabel("Persentase (%)")
axes[1].tick_params(axis="x", rotation=0)
save_plot("02_distribusi_target.png")

# Visualisasi makronutrien terhadap kategori target.
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
for ax, col in zip(axes, MACRO_COLS):
    sns.boxplot(data=df, x="kategori_kalori", y=col,
                order=TARGET_ORDER, color="steelblue", ax=ax)
    ax.set_title(f"{col} menurut kategori kalori")
    ax.set_xlabel("Kategori kalori")
    ax.set_ylabel("Gram (unit dataset)")
save_plot("03_boxplot_fitur_vs_target.png")

mean_by_target = df.groupby("kategori_kalori")[MACRO_COLS].mean().reindex(TARGET_ORDER)
mean_by_target.to_csv(OUT_DIR / "02_rata_rata_makro_per_target.csv")
print("\nRata-rata makronutrien per kategori target:")
print(mean_by_target.round(2))



[CAPAIAN 7] DISTRIBUSI TARGET
kategori_kalori
Rendah    684
Tinggi    387
Sedang    275
Name: count, dtype: int64

Rata-rata makronutrien per kategori target:
                 proteins    fat  carbohydrate
kategori_kalori                               
Rendah               5.76   1.88         10.70
Sedang              13.90   6.96         24.53
Tinggi              14.73  18.11         51.96


## 8. EDA: KORELASI DAN INSIGHT

In [8]:
print("\n[CAPAIAN 8] KORELASI PEARSON")
correlation = df[NUM_COLS].corr(method="pearson")
correlation.to_csv(OUT_DIR / "03_matriks_korelasi_pearson.csv")
plt.figure(figsize=(8, 6))
sns.heatmap(correlation, annot=True, cmap="coolwarm", fmt=".2f",
            vmin=-1, vmax=1, square=True)
plt.title("Heatmap Korelasi Pearson Fitur Gizi")
save_plot("04_heatmap_korelasi.png")

# Korelasi Spearman antara makro dan target ordinal sebagai informasi tambahan.
target_ordinal = df["kategori_kalori"].map(TARGET_MAP)
spearman_target = df[MACRO_COLS].corrwith(target_ordinal, method="spearman")
print("Korelasi Spearman makronutrien terhadap target ordinal:")
print(spearman_target.round(3))

# Cetak tiga insight berdasarkan hasil aktual, bukan angka yang diketik manual.
print("\nINSIGHT KORELASI:")
cal_corr = correlation["calories"].drop("calories").sort_values(key=lambda s: s.abs(), ascending=False)
print(f"1. Hubungan linear terkuat dengan kalori adalah {cal_corr.index[0]} "
      f"(Pearson r={cal_corr.iloc[0]:.3f}).")
macro_corr = df[MACRO_COLS].corr()
pairs = [(a, b, macro_corr.loc[a, b]) for i, a in enumerate(MACRO_COLS)
         for b in MACRO_COLS[i + 1:]]
pairs.sort(key=lambda x: abs(x[2]))
print(f"2. Korelasi antarmakro terlemah adalah {pairs[0][0]} dengan {pairs[0][1]} "
      f"(r={pairs[0][2]:.3f}); periksa pasangan dengan korelasi absolut tertinggi "
      f"sebelum menyimpulkan redundansi fitur.")
print(f"3. Korelasi Spearman terkuat terhadap urutan target adalah "
      f"{spearman_target.abs().idxmax()} (rho={spearman_target.loc[spearman_target.abs().idxmax()]:.3f}).")
print("Catatan: korelasi tidak membuktikan sebab-akibat.")



[CAPAIAN 8] KORELASI PEARSON
Korelasi Spearman makronutrien terhadap target ordinal:
proteins        0.336
fat             0.533
carbohydrate    0.498
dtype: float64

INSIGHT KORELASI:
1. Hubungan linear terkuat dengan kalori adalah fat (Pearson r=0.738).
2. Korelasi antarmakro terlemah adalah fat dengan carbohydrate (r=-0.004); periksa pasangan dengan korelasi absolut tertinggi sebelum menyimpulkan redundansi fitur.
3. Korelasi Spearman terkuat terhadap urutan target adalah fat (rho=0.533).
Catatan: korelasi tidak membuktikan sebab-akibat.


## 9. AUDIT KUALITAS: MISSING, OUTLIER IQR, DUPLIKASI PER KOLOM

In [9]:
print("\n[CAPAIAN 9] AUDIT KUALITAS DATA")

def iqr_audit(series):
    s = series.dropna()
    if s.empty:
        return 0, np.nan, np.nan
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    count = int(((s < lower) | (s > upper)).sum())
    return count, lower, upper

audit_rows = []
for col in df.columns:
    row = {
        "kolom": col,
        "tipe_data": str(df[col].dtype),
        "missing": int(df[col].isna().sum()),
        "persen_missing": round(df[col].isna().mean() * 100, 2),
        "jumlah_nilai_berulang": int(df[col].duplicated().sum()),
    }
    if col in NUM_COLS:
        n_out, low, high = iqr_audit(df[col])
        row.update({"outlier_IQR": n_out,
                    "persen_outlier_IQR": round(n_out / max(df[col].notna().sum(), 1) * 100, 2),
                    "batas_bawah_IQR": low,
                    "batas_atas_IQR": high})
    else:
        row.update({"outlier_IQR": "-", "persen_outlier_IQR": "-",
                    "batas_bawah_IQR": "-", "batas_atas_IQR": "-"})
    audit_rows.append(row)

audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv(OUT_DIR / "04_audit_kualitas_per_kolom.csv", index=False)
print(audit_df.to_string(index=False))
print("Duplikat baris penuh:", int(df.duplicated().sum()))
print("Duplikat nama (abaikan kapital/spasi):",
      int(df["name"].str.lower().duplicated().sum()))

# Simpan daftar outlier IQR per kolom numerik untuk inspeksi.
outlier_parts = []
for col in NUM_COLS:
    n_out, low, high = iqr_audit(df[col])
    mask = df[col].notna() & ((df[col] < low) | (df[col] > high))
    part = df.loc[mask, ["id", "name", col]].copy()
    if not part.empty:
        part["variabel"] = col
        part["batas_bawah_IQR"] = low
        part["batas_atas_IQR"] = high
        part = part.rename(columns={col: "nilai_outlier"})
        outlier_parts.append(part)
if outlier_parts:
    pd.concat(outlier_parts, ignore_index=True).to_csv(
        OUT_DIR / "05_daftar_outlier_IQR.csv", index=False)



[CAPAIAN 9] AUDIT KUALITAS DATA
          kolom tipe_data  missing  persen_missing  jumlah_nilai_berulang outlier_IQR persen_outlier_IQR batas_bawah_IQR batas_atas_IQR
             id     int64        0             0.0                      0           -                  -               -              -
       calories   float64        0             0.0                    874          13               0.97        -313.125        721.875
       proteins   float64        0             0.0                   1034          64               4.75           -18.0           34.8
            fat   float64        0             0.0                   1067         162              12.04        -11.1625        19.9375
   carbohydrate   float64        0             0.0                    802          17               1.26          -45.05          87.15
           name    string        0             0.0                      0           -                  -               -              -
          image

## 10. DATA CLEANING

In [10]:
print("\n[CAPAIAN 10] DATA CLEANING")
df_before = df.copy()
df_clean = df.copy()
clean_log = []

# A. Rapikan teks permanen.
for col in ["name", "image"]:
    df_clean[col] = df_clean[col].astype("string").str.strip()

# B. Missing value: calories harus ada karena dipakai membentuk target.
# Nutrisi yang kosong diimputasi median kolom; baris tanpa nama dibuang.
start = len(df_clean)
df_clean = df_clean.dropna(subset=["calories", "name"])
clean_log.append(("Hapus baris tanpa calories atau name", start - len(df_clean)))

for col in MACRO_COLS:
    n_missing = int(df_clean[col].isna().sum())
    if n_missing:
        median_value = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(median_value)
    clean_log.append((f"Imputasi missing {col} dengan median", n_missing))

# C. Hapus duplikasi nama tanpa membedakan kapitalisasi.
start = len(df_clean)
nama_key = df_clean["name"].str.casefold()
df_clean = df_clean.loc[~nama_key.duplicated(keep="first")].copy()
clean_log.append(("Hapus nama duplikat (case-insensitive setelah strip)", start - len(df_clean)))

# D. Hapus baris dengan jumlah makronutrien >100, sesuai asumsi basis 100 g.
# Validitas basis 100 g harus tetap dinyatakan sebagai asumsi jika metadata sumber tidak menyebut unit.
start = len(df_clean)
macro_total = df_clean[MACRO_COLS].sum(axis=1)
df_clean = df_clean.loc[macro_total <= 100].copy()
clean_log.append(("Hapus total makronutrien >100", start - len(df_clean)))

# E. Tangani ketidaksesuaian energi. Rumus Atwater sederhana adalah pemeriksaan indikatif,
# bukan hukum mutlak; perbedaan dapat muncul karena serat, faktor energi spesifik, pembulatan,
# dan definisi karbohidrat. Sesuai aturan proyek, baris dengan selisih relatif >50%
# DAN selisih absolut >20 kkal ditandai lalu dikeluarkan. Daftarnya disimpan untuk audit.
energy_est = (4 * df_clean["proteins"] + 9 * df_clean["fat"]
              + 4 * df_clean["carbohydrate"])
abs_diff = (df_clean["calories"] - energy_est).abs()
relative_diff = abs_diff / energy_est.clip(lower=1)
energy_flag = (relative_diff > 0.50) & (abs_diff > 20)
flagged_energy = df_clean.loc[energy_flag, ["id", "name", *NUM_COLS]].copy()
flagged_energy["estimasi_Atwater"] = energy_est.loc[energy_flag]
flagged_energy["selisih_absolut"] = abs_diff.loc[energy_flag]
flagged_energy["selisih_relatif_persen"] = relative_diff.loc[energy_flag] * 100
flagged_energy.to_csv(OUT_DIR / "06_baris_kalori_perlu_verifikasi.csv", index=False)
start = len(df_clean)
df_clean = df_clean.loc[~energy_flag].copy()
clean_log.append(("Hapus selisih energi >50% dan >20 kkal", start - len(df_clean)))

df_clean = df_clean.reset_index(drop=True)
# Buat ulang target setelah cleaning agar konsisten dengan data final.
df_clean["kategori_kalori"] = df_clean["calories"].apply(kategori_kalori)

clean_log_df = pd.DataFrame(clean_log, columns=["langkah_cleaning", "jumlah_baris_terdampak"])
clean_log_df.to_csv(OUT_DIR / "07_log_cleaning.csv", index=False)
print(clean_log_df.to_string(index=False))
print(f"Jumlah baris awal: {len(df_before)}")
print(f"Jumlah baris setelah cleaning: {len(df_clean)}")
print(f"Total baris yang dibuang: {len(df_before) - len(df_clean)}")
print("Missing value setelah cleaning:")
print(df_clean.isna().sum())

if df_clean.empty:
    raise ValueError("Data menjadi kosong setelah cleaning. Periksa kembali aturan cleaning.")
if df_clean["kategori_kalori"].nunique() < 2:
    raise ValueError("Kategori target kurang dari dua kelas; tidak dapat melakukan klasifikasi.")



[CAPAIAN 10] DATA CLEANING
                                    langkah_cleaning  jumlah_baris_terdampak
                Hapus baris tanpa calories atau name                       0
             Imputasi missing proteins dengan median                       0
                  Imputasi missing fat dengan median                       0
         Imputasi missing carbohydrate dengan median                       0
Hapus nama duplikat (case-insensitive setelah strip)                       3
                       Hapus total makronutrien >100                      11
              Hapus selisih energi >50% dan >20 kkal                      27
Jumlah baris awal: 1346
Jumlah baris setelah cleaning: 1305
Total baris yang dibuang: 41
Missing value setelah cleaning:
id                 0
calories           0
proteins           0
fat                0
carbohydrate       0
name               0
image              0
kategori_kalori    0
dtype: int64


## 11. PERBANDINGAN SEBELUM VS SESUDAH CLEANING

In [11]:
def ringkasan_kondisi(data, label):
    total_macro = data[MACRO_COLS].sum(axis=1)
    energy = 4 * data["proteins"] + 9 * data["fat"] + 4 * data["carbohydrate"]
    abs_d = (data["calories"] - energy).abs()
    rel_d = abs_d / energy.clip(lower=1)
    return pd.Series({
        "jumlah_baris": len(data),
        "missing_total": int(data[required].isna().sum().sum()),
        "duplikat_baris": int(data[required].duplicated().sum()),
        "nama_duplikat_normalisasi": int(data["name"].str.strip().str.lower().duplicated().sum()),
        "total_makro_di_atas_100": int((total_macro > 100).sum()),
        "kalori_tidak_konsisten": int(((rel_d > 0.50) & (abs_d > 20)).sum()),
        "maksimum_karbohidrat": data["carbohydrate"].max(),
    }, name=label)

comparison = pd.concat([
    ringkasan_kondisi(df_before, "Sebelum"),
    ringkasan_kondisi(df_clean, "Sesudah")
], axis=1)
comparison.to_csv(OUT_DIR / "08_perbandingan_sebelum_sesudah.csv")
print("\n[CAPAIAN 11] PERBANDINGAN SEBELUM VS SESUDAH")
print(comparison)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for j, col in enumerate(MACRO_COLS):
    sns.boxplot(y=df_before[col], color="salmon", ax=axes[0, j])
    axes[0, j].set_title(f"{col} - Sebelum")
    sns.boxplot(y=df_clean[col], color="seagreen", ax=axes[1, j])
    axes[1, j].set_title(f"{col} - Sesudah")
axes[0, 2].set_ylabel("Nilai")
axes[1, 2].set_ylabel("Nilai")
save_plot("05_boxplot_sebelum_sesudah.png")



[CAPAIAN 11] PERBANDINGAN SEBELUM VS SESUDAH
                           Sebelum  Sesudah
jumlah_baris                1346.0   1305.0
missing_total                  0.0      0.0
duplikat_baris                 0.0      0.0
nama_duplikat_normalisasi      3.0      0.0
total_makro_di_atas_100       11.0      0.0
kalori_tidak_konsisten        30.0      0.0
maksimum_karbohidrat         647.0     95.3


## 12. PREPROCESSING: FEATURE ENGINEERING, ENCODING, SCALING, IMBALANCE

In [12]:
print("\n[CAPAIAN 12] PREPROCESSING")
# Dataset asli tidak mempunyai kolom metode_olah dan kelompok_pangan.
# Keduanya diturunkan secara heuristik dari nama makanan, sehingga merupakan
# fitur aproksimasi dan bukan kategori resmi TKPI.
def metode_olah(nama):
    n = str(nama).lower()
    if "goreng" in n:
        return "goreng"
    if "rebus" in n or "kukus" in n:
        return "rebus_kukus"
    if "bakar" in n or "panggang" in n:
        return "bakar_panggang"
    if "masakan" in n or "masak " in n:
        return "masakan"
    if "kering" in n:
        return "kering"
    if "segar" in n or "mentah" in n:
        return "segar_mentah"
    return "tidak_diketahui_dari_nama"

KEYWORDS = {
    "minyak_lemak": ["minyak", "lemak", "mentega", "margarin"],
    "ikan_seafood": ["ikan", "udang", "kerang", "cumi", "kepiting", "tiram", "teri", "rebon", "belut"],
    "daging_unggas": ["daging", "ayam", "sapi", "kambing", "bebek", "itik", "usus", "babat", "hati", "dendeng", "sosis", "abon"],
    "telur_susu": ["telur", "susu", "keju", "yoghurt"],
    "kacang_olahan": ["kacang", "kedelai", "tempe", "tahu", "koro", "biji", "oncom"],
    "sayur": ["daun", "sayur", "kangkung", "bayam", "kol", "terong", "tauge", "sawi", "labu", "buncis"],
    "buah": ["buah", "jeruk", "pisang", "mangga", "apel", "salak", "sirsak", "pepaya", "nanas", "semangka", "durian", "jambu", "rambutan", "alpukat", "anggur", "melon", "nangka"],
    "umbi_serealia": ["ubi", "singkong", "talas", "kentang", "sagu", "beras", "nasi", "jagung", "gandum", "mie", "mi ", "roti", "tepung", "bihun", "ketan"],
    "jajanan_olahan": ["kue", "kerupuk", "keripik", "bika", "dodol", "wajik", "biskuit", "cake", "opak", "pilus", "bubur", "bakso", "lemper", "onde"],
}

def kelompok_pangan(nama):
    n = str(nama).lower()
    for category, words in KEYWORDS.items():
        if any(word in n for word in words):
            return category
    return "lainnya_tidak_teridentifikasi"

df_clean["metode_olah"] = df_clean["name"].map(metode_olah)
df_clean["kelompok_pangan"] = df_clean["name"].map(kelompok_pangan)
df_clean[["id", "name", "metode_olah", "kelompok_pangan"]].to_csv(
    OUT_DIR / "09_fitur_kategorikal_turunan.csv", index=False)

# Target ordinal. Jangan masukkan calories ke X karena target berasal dari calories.
y = df_clean["kategori_kalori"].map(TARGET_MAP).astype(int)
feature_num = MACRO_COLS
feature_cat = ["metode_olah", "kelompok_pangan"]
X = df_clean[feature_num + feature_cat].copy()

print("Fitur numerik:", feature_num)
print("Fitur kategorikal turunan:", feature_cat)
print("Kolom calories dikeluarkan dari X untuk mencegah target leakage.")



[CAPAIAN 12] PREPROCESSING
Fitur numerik: ['proteins', 'fat', 'carbohydrate']
Fitur kategorikal turunan: ['metode_olah', 'kelompok_pangan']
Kolom calories dikeluarkan dari X untuk mencegah target leakage.


## 13. SPLIT TRAIN/TEST; FIT TRANSFORMASI PADA TRAIN SAJA

In [13]:
print("\n[CAPAIAN 13] TRAIN-TEST SPLIT")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y)
print("Jumlah train:", len(X_train), "| Jumlah test:", len(X_test))

# Kompatibilitas scikit-learn versi baru/lama untuk OneHotEncoder.
try:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", RobustScaler(), feature_num),
        ("cat", encoder, feature_cat),
    ],
    remainder="drop"
)

X_train_array = preprocessor.fit_transform(X_train)
X_test_array = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()
feature_names = [name.replace("num__", "").replace("cat__", "") for name in feature_names]
X_train_encoded = pd.DataFrame(X_train_array, columns=feature_names, index=X_train.index)
X_test_encoded = pd.DataFrame(X_test_array, columns=feature_names, index=X_test.index)

print("Jumlah fitur setelah encoding:", X_train_encoded.shape[1])
print("Distribusi target train:")
print(y_train.map({0: "Rendah", 1: "Sedang", 2: "Tinggi"}).value_counts().reindex(TARGET_ORDER))
print("Distribusi target test:")
print(y_test.map({0: "Rendah", 1: "Sedang", 2: "Tinggi"}).value_counts().reindex(TARGET_ORDER))

# Oversampling manual hanya pada data latih, setelah encoding dan scaling.
# Data test tidak diubah agar evaluasi nantinya tetap representatif.
train_frame = X_train_encoded.copy()
train_frame["__target__"] = y_train
max_class_count = int(y_train.value_counts().max())
resampled_parts = []
for cls in sorted(y_train.unique()):
    class_part = train_frame[train_frame["__target__"] == cls]
    sampled = class_part.sample(n=max_class_count, replace=True, random_state=RANDOM_STATE)
    resampled_parts.append(sampled)
train_balanced = pd.concat(resampled_parts, ignore_index=True)
y_train_balanced = train_balanced.pop("__target__").astype(int)
X_train_balanced = train_balanced

print("Distribusi train setelah random oversampling:")
print(y_train_balanced.map({0: "Rendah", 1: "Sedang", 2: "Tinggi"}).value_counts().reindex(TARGET_ORDER))



[CAPAIAN 13] TRAIN-TEST SPLIT
Jumlah train: 1044 | Jumlah test: 261
Jumlah fitur setelah encoding: 20
Distribusi target train:
kategori_kalori
Rendah    535
Sedang    213
Tinggi    296
Name: count, dtype: int64
Distribusi target test:
kategori_kalori
Rendah    134
Sedang     53
Tinggi     74
Name: count, dtype: int64
Distribusi train setelah random oversampling:
__target__
Rendah    535
Sedang    535
Tinggi    535
Name: count, dtype: int64


## 14. FEATURE SELECTION: MUTUAL INFORMATION, ANOVA, RANDOM FOREST

In [14]:
print("\n[CAPAIAN 14] FEATURE SELECTION")
# MI dan ANOVA dihitung pada train asli (bukan test dan bukan data oversampling).
mi_scores = mutual_info_classif(
    X_train_encoded, y_train, random_state=RANDOM_STATE)
f_scores, p_values = f_classif(X_train_encoded, y_train)

# Random Forest dilatih pada train seimbang; class_weight juga menjaga perhatian kelas.
rf = RandomForestClassifier(
    n_estimators=400,
    random_state=RANDOM_STATE,
    class_weight="balanced_subsample",
    min_samples_leaf=2,
    n_jobs=-1
)
rf.fit(X_train_balanced, y_train_balanced)

selection = pd.DataFrame({
    "fitur": X_train_encoded.columns,
    "mutual_information": mi_scores,
    "f_score_anova": f_scores,
    "p_value_anova": p_values,
    "random_forest_importance": rf.feature_importances_,
}).sort_values("random_forest_importance", ascending=False)
selection.to_csv(OUT_DIR / "10_hasil_feature_selection.csv", index=False)
print(selection.round(4).to_string(index=False))

# Visualisasi 12 fitur dengan importance tertinggi.
top_rf = selection.nlargest(12, "random_forest_importance")
plt.figure(figsize=(9, 6))
sns.barplot(data=top_rf, y="fitur", x="random_forest_importance", color="steelblue")
plt.title("Top Feature Importance - Random Forest")
plt.xlabel("Feature importance")
plt.ylabel("Fitur")
save_plot("06_feature_importance_random_forest.png")

top_mi = selection.nlargest(12, "mutual_information")
plt.figure(figsize=(9, 6))
sns.barplot(data=top_mi, y="fitur", x="mutual_information", color="darkorange")
plt.title("Top Feature Importance - Mutual Information")
plt.xlabel("Mutual information")
plt.ylabel("Fitur")
save_plot("07_feature_importance_mutual_information.png")

# Ambang seleksi dibuat eksplisit dan dapat dijelaskan.
selected_features = selection.loc[
    (selection["mutual_information"] > 0.01)
    | (selection["random_forest_importance"] > 0.01),
    "fitur"
].tolist()
if not selected_features:
    # Pengaman: bila tidak ada fitur melewati ambang, ambil tiga fitur teratas RF.
    selected_features = selection.nlargest(3, "random_forest_importance")["fitur"].tolist()

removed_features = [f for f in X_train_encoded.columns if f not in selected_features]
print("\nFitur terpilih:", selected_features)
print("Fitur yang tidak terpilih:", removed_features)

X_train_final = X_train_balanced[selected_features].copy()
y_train_final = y_train_balanced.copy()
X_test_final = X_test_encoded[selected_features].copy()
y_test_final = y_test.copy()

# Simpan matriks final agar dapat digunakan pada tahap modeling berikutnya.
X_train_final.to_csv(OUT_DIR / "11_X_train_final.csv", index=False)
y_train_final.to_csv(OUT_DIR / "12_y_train_final.csv", index=False, header=["target"])
X_test_final.to_csv(OUT_DIR / "13_X_test_final.csv", index=False)
y_test_final.to_csv(OUT_DIR / "14_y_test_final.csv", index=False, header=["target"])
selection.to_csv(OUT_DIR / "15_ranking_fitur.csv", index=False)

# Simpan data bersih beserta fitur turunannya.
df_clean.to_csv(OUT_DIR / "16_dataset_bersih.csv", index=False)

print("\n" + "=" * 65)
print("SELESAI: PROGRESS 2 (CAPAIAN 6-14)")
print("Data awal:", len(df_before), "| Data bersih:", len(df_clean))
print("Fitur akhir:", len(selected_features))
print("Seluruh tabel, dataset, dan grafik tersimpan di:", OUT_DIR)
print("=" * 65)



[CAPAIAN 14] FEATURE SELECTION
                                        fitur  mutual_information  f_score_anova  p_value_anova  random_forest_importance
                                 carbohydrate              0.3741       375.4788         0.0000                    0.3704
                                          fat              0.2961       196.0619         0.0000                    0.2751
                                     proteins              0.1479        86.4619         0.0000                    0.1691
                     metode_olah_segar_mentah              0.0380        43.7425         0.0000                    0.0291
                        kelompok_pangan_sayur              0.0646        50.7804         0.0000                    0.0254
        metode_olah_tidak_diketahui_dari_nama              0.0191         8.9187         0.0001                    0.0160
                kelompok_pangan_umbi_serealia              0.0038        22.4237         0.0000                   